# Global Macroeconomics

Cross-country comparisons for the economies tracked in this folder. Nominal GDP is measured in current U.S. dollars so values are comparable and additive across economies.

In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / 'Quantapp' / '__init__.py').is_file():
        project_root = candidate
        break
else:
    raise RuntimeError('Could not locate the Investment Research project root')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display
from Quantapp.data import fetch_nominal_gdp_current_usd


In [ ]:
# World Bank economy codes for every country/region notebook in this folder.
tracked_economies = {
    'AUS': 'Australia',
    'BRA': 'Brazil',
    'CAN': 'Canada',
    'CHN': 'China',
    'EMU': 'Euro Area',
    'IND': 'India',
    'JPN': 'Japan',
    'MEX': 'Mexico',
    'SAU': 'Saudi Arabia',
    'KOR': 'South Korea',
    'GBR': 'United Kingdom',
    'USA': 'United States',
}

nominal_gdp_usd = fetch_nominal_gdp_current_usd(tracked_economies)
latest_year_by_economy = nominal_gdp_usd.apply(lambda series: series.last_valid_index())
display(pd.DataFrame({'Latest available year': latest_year_by_economy}))

# Use the common-history panel so every stacked total contains all economies.
nominal_gdp_common = nominal_gdp_usd.dropna(how='any').div(1e12)
# Fix the visual order using the latest complete year: largest at the top.
descending_order = (
    nominal_gdp_common.iloc[-1]
    .sort_values(ascending=False)
    .index.tolist()
)
stack_order = list(reversed(descending_order))
nominal_gdp_common = nominal_gdp_common.loc[:, stack_order]
nominal_gdp_share = nominal_gdp_common.div(nominal_gdp_common.sum(axis=1), axis=0).mul(100)
nominal_gdp_share_long = (
    nominal_gdp_share
    .rename_axis('Year')
    .reset_index()
    .melt(id_vars='Year', var_name='Economy', value_name='GDP Share')
)

fig = px.area(
    nominal_gdp_share_long,
    x='Year',
    y='GDP Share',
    color='Economy',
    category_orders={'Economy': stack_order},
    title='Share of Tracked Nominal GDP by Economy',
    labels={'GDP Share': 'Share of tracked nominal GDP (%)'},
)
trace_names = [trace.name for trace in fig.data]
percentage_values = [nominal_gdp_share[name].to_numpy() for name in trace_names]
usd_values = [nominal_gdp_common[name].to_numpy() for name in trace_names]
percentage_hover = [
    '%{fullData.name}<br>Year: %{x}<br>Share: %{y:.2f}%<extra></extra>'
    for _ in trace_names
]
usd_hover = [
    '%{fullData.name}<br>Year: %{x}<br>Nominal GDP: $%{y:.2f}T<extra></extra>'
    for _ in trace_names
]
flag_by_economy = {
    'Australia': '🇦🇺', 'Brazil': '🇧🇷', 'Canada': '🇨🇦',
    'China': '🇨🇳', 'Euro Area': '🇪🇺', 'India': '🇮🇳',
    'Japan': '🇯🇵', 'Mexico': '🇲🇽', 'Saudi Arabia': '🇸🇦',
    'South Korea': '🇰🇷', 'United Kingdom': '🇬🇧',
    'United States': '🇺🇸',
}

def build_flag_annotations(frame):
    latest = frame.iloc[-1]
    midpoints = latest.cumsum() - latest.div(2)
    return [
        dict(
            x=frame.index[-1] + 0.5,
            y=midpoints[economy],
            text=flag_by_economy[economy],
            showarrow=False,
            xanchor='left',
            yanchor='middle',
            font=dict(size=16),
        )
        for economy in frame.columns
    ]

percentage_flag_annotations = build_flag_annotations(nominal_gdp_share)
usd_flag_annotations = build_flag_annotations(nominal_gdp_common)
fig.update_layout(
    template='plotly_dark',
    hovermode='x unified',
    height=750,
    annotations=percentage_flag_annotations,
    margin=dict(r=75),
    legend=dict(title_text='Economy', traceorder='reversed'),
    updatemenus=[dict(
        type='dropdown',
        direction='down',
        active=0,
        x=1.0,
        xanchor='right',
        y=1.14,
        yanchor='top',
        buttons=[
            dict(
                label='Percentage share',
                method='update',
                args=[
                    {'y': percentage_values, 'hovertemplate': percentage_hover},
                    {
                        'title.text': 'Share of Tracked Nominal GDP by Economy',
                        'yaxis.title.text': 'Share of tracked nominal GDP (%)',
                        'yaxis.tickprefix': '',
                        'yaxis.ticksuffix': '%',
                        'yaxis.tickformat': '.1f',
                        'annotations': percentage_flag_annotations,
                    },
                ],
            ),
            dict(
                label='USD trillions',
                method='update',
                args=[
                    {'y': usd_values, 'hovertemplate': usd_hover},
                    {
                        'title.text': 'Nominal GDP by Economy (Current U.S. Dollars)',
                        'yaxis.title.text': 'Nominal GDP (USD trillions)',
                        'yaxis.tickprefix': '$',
                        'yaxis.ticksuffix': 'T',
                        'yaxis.tickformat': '.2f',
                        'annotations': usd_flag_annotations,
                    },
                ],
            ),
        ],
    )],
)
for trace, hovertemplate in zip(fig.data, percentage_hover):
    trace.hovertemplate = hovertemplate
fig.update_yaxes(tickprefix='', ticksuffix='%', tickformat='.1f')
fig.update_xaxes(range=[nominal_gdp_common.index.min(), nominal_gdp_common.index.max() + 2])
fig.show()

display(Markdown(
    f"World Bank indicator: `NY.GDP.MKTP.CD` (GDP, current US$). "
    f"API last updated: **{nominal_gdp_usd.attrs.get('last_updated', 'unknown')}**. "
    f"The complete comparison spans **{nominal_gdp_common.index.min()}–{nominal_gdp_common.index.max()}**."
))
